# Data Preparation
## Reddit r/politics -- 2024 US Presidential Election

Loads, filters, and joins raw Reddit JSONL files into two clean DataFrames:
- **`df_posts`** -- election-relevant posts (June--November 2024)
- **`df_comments`** -- comments joined to their parent post via `link_id`

Saves cleaned data as Parquet to the `Datasources/` folder for use in the analysis notebook.

**Folder structure:**
```
Final project/
├── Notebooks/
│   └── Data_preparation.ipynb   <- you are here
├── Datasources/                 <- all data files live here
│   ├── *.jsonl                  <- raw JSONL (not on GitHub)
│   ├── 2024_Election_Polls.xlsx
│   ├── df_posts_filtered.parquet    <- output
│   └── df_comments_filtered.parquet <- output
└── README.md
```

> **Teammates:** rename your downloaded Google Drive folder to exactly `Datasources` and place it inside `Final project/`. No path changes needed.

## 1. Imports

In [1]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path('..').resolve()      # project root (one level up from Notebooks/)
DATA_DIR = BASE_DIR / 'Datasources' # all raw + processed data

print('Project root:', BASE_DIR)
print('Data folder: ', DATA_DIR)
assert DATA_DIR.exists(), f"Datasources/ folder not found at {DATA_DIR}"

Project root: C:\Users\Dell\Desktop\NLP\Final project
Data folder:  C:\Users\Dell\Desktop\NLP\Final project\Datasources


## 2. Election Keyword Filter

~26% of r/politics posts are off-topic (spam, unrelated news).  
We keep only posts whose title contains at least one election keyword.

In [2]:
ELECTION_KEYWORDS = [
    # Candidates
    'trump', 'harris', 'kamala', 'biden', 'walz', 'vance', 'jd vance', 'maga', 'donald',
    # Election mechanics
    'election', 'vote', 'voting', 'ballot', 'poll', 'polling',
    'electoral', 'swing state', 'battleground', 'campaign',
    # Parties & events
    'democrat', 'republican', 'gop', 'dnc', 'rnc', 'debate',
    # Policy
    'project 2025', 'abortion', 'white house', 'oval office',
]

def is_election_related(title: str) -> bool:
    tl = title.lower()
    return any(kw in tl for kw in ELECTION_KEYWORDS)

print(f'Filter has {len(ELECTION_KEYWORDS)} keywords')

Filter has 29 keywords


## 3. Load and Filter Posts

One JSONL file per month. Each line is one Reddit post.  
Key columns: `id`, `name` (join key for comments), `title`, `score`, `date`.

In [3]:
POST_FILES = {
    'June':      'r_politics_posts_June_combined.jsonl',
    'July':      'r_politics_posts_July_combined.jsonl',
    'August':    'r_politics_posts_August_combined.jsonl',
    'September': 'r_politics_posts_September_combined.jsonl',
    'October':   'r_politics_posts_October_combined.jsonl',
    'November':  'r_politics_posts_November_combined.jsonl',
}

all_posts = []
for month, fname in POST_FILES.items():
    fpath = DATA_DIR / fname
    if not fpath.exists():
        print(f'{month:10s}: FILE NOT FOUND -- skipping ({fname})')
        continue
    raw, kept = 0, 0
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        for line in f:
            p = json.loads(line)
            raw += 1
            if is_election_related(p.get('title', '')):
                p['month'] = month
                all_posts.append(p)
                kept += 1
    print(f'{month:10s}: {raw:6,} total  ->  {kept:6,} kept  ({kept/raw*100:.1f}%)')

df_posts = pd.DataFrame(all_posts)[[
    'id', 'name', 'title', 'selftext', 'score',
    'upvote_ratio', 'num_comments', 'created_utc', 'month'
]].copy()

df_posts['date']     = pd.to_datetime(df_posts['created_utc'], unit='s')
df_posts['date_day'] = df_posts['date'].dt.date
df_posts['selftext'] = df_posts['selftext'].replace({'[removed]': '', '[deleted]': ''}).fillna('')
df_posts = df_posts.drop(columns=['created_utc'])

print(f'\nTotal posts: {len(df_posts):,}')
print(f'Date range:  {df_posts["date"].min().date()} -> {df_posts["date"].max().date()}')
df_posts.head()

June      :  9,820 total  ->   5,951 kept  (60.6%)
July      : 20,816 total  ->  15,931 kept  (76.5%)
August    : 17,764 total  ->  13,207 kept  (74.3%)
September : 14,596 total  ->  10,682 kept  (73.2%)
October   : 16,183 total  ->  12,388 kept  (76.5%)
November  :  6,562 total  ->   5,082 kept  (77.4%)

Total posts: 63,241
Date range:  2024-06-01 -> 2024-11-06


,id,name,title,selftext,score,upvote_ratio,num_comments,month,date,date_day
0,1d5a0m0,t3_1d5a0m0,How has the recent controversy around Trump's ...,,1,1.00,0,June,2024-06-01 00:03:11,2024-06-01
1,1d5a8ie,t3_1d5a8ie,r/politics The hush money mess with ex-POTUS...,,1,1.00,0,June,2024-06-01 00:13:59,2024-06-01
2,1d5a9uj,t3_1d5a9uj,Trump Hoped 'My Juror' Would Save Him From Con...,,12941,0.95,1550,June,2024-06-01 00:15:56,2024-06-01
3,1d5aazn,t3_1d5aazn,Investigations into Donald Trump,,1,1.00,0,June,2024-06-01 00:17:34,2024-06-01
4,1d5adpv,t3_1d5adpv,Here's what you should know about Donald Trump...,,1,1.00,1,June,2024-06-01 00:21:32,2024-06-01


In [4]:
print('Posts per month:')
print(df_posts['month'].value_counts().reindex(
    ['June','July','August','September','October','November']
))

Posts per month:
month
June          5951
July         15931
August       13207
September    10682
October      12388
November      5082
Name: count, dtype: int64


## 4. Load Comments and Join to Posts

Each comment has a `link_id` (e.g. `t3_1eh2l76`) that always points to the root post,  
regardless of nesting depth -- even deeply nested replies trace back to the original post.

Strategy:
1. Read line by line, filter during loading (avoids holding millions of rows in memory)
2. Use `orjson` for faster JSON parsing (3-5x speedup over standard `json`)
3. Skip `post_title` during loading -- attach via pandas left join after
4. Remove bot/automod comments before saving

> **Run once, save to Parquet, never repeat.**

In [5]:
try:
    import orjson
    loads = orjson.loads
    print('Using orjson (fast)')
except ImportError:
    loads = json.loads
    print('orjson not found -- using standard json. Install with: pip install orjson')

Using orjson (fast)


In [ ]:
valid_post_names = set(df_posts['name'])

COMMENT_FILES = {
    'June':      'r_politics_comments_June_combined.jsonl',
    'July':      'r_politics_comments_July_combined.jsonl',
    'August':    'r_politics_comments_August_combined.jsonl',
    'September': 'r_politics_comments_September_combined.jsonl',
    'October':   'r_politics_comments_October_combined.jsonl',
    'November':  'r_politics_comments_November_combined.jsonl',
}

all_comments = []
for month, fname in COMMENT_FILES.items():
    fpath = DATA_DIR / fname
    if not fpath.exists():
        print(f'{month:10s}: FILE NOT FOUND -- skipping ({fname})')
        continue
    raw, kept = 0, 0
    with open(fpath, encoding='utf-8', errors='ignore') as f:
        for line in f:
            c = loads(line)
            raw += 1
            link_id = c.get('link_id', '')
            body    = c.get('body', '')
            if link_id in valid_post_names and body not in ('', '[removed]', '[deleted]'):
                all_comments.append({
                    'comment_id':   c['id'],
                    'post_id':      link_id,
                    'parent_id':    c.get('parent_id', ''),
                    'is_top_level': c.get('parent_id', '').startswith('t3_'),
                    'body':         body,
                    'score':        c.get('score', 0),
                    'date':         pd.to_datetime(c['created_utc'], unit='s'),
                    'month':        month,
                })
                kept += 1
    print(f'{month:10s}: {raw:8,} total  ->  {kept:7,} kept  ({kept/raw*100:.1f}%)')

df_comments = pd.DataFrame(all_comments)
df_comments['date_day'] = df_comments['date'].dt.date
print(f'\nTotal comments loaded: {len(df_comments):,}')

June      :  806,583 total  ->  615,592 kept  (76.3%)


In [7]:
# Left join post_title from df_posts
df_comments = df_comments.merge(
    df_posts[['name', 'title']].rename(columns={'name': 'post_id', 'title': 'post_title'}),
    on='post_id',
    how='left'
)

print(f'Shape after join: {df_comments.shape}')
df_comments.head()

Shape after join: (6652222, 10)


,comment_id,post_id,parent_id,is_top_level,body,score,date,month,date_day,post_title
0,l6k03bu,t3_1d5a9uj,t3_1d5a9uj,True,"\nAs a reminder, this subreddit [is for civil ...",1,2024-06-01 00:15:56,June,2024-06-01,Trump Hoped 'My Juror' Would Save Him From Con...
1,l6k03do,t3_1d5a9uj,t3_1d5a9uj,True,This submission source is likely to have a sof...,1,2024-06-01 00:15:57,June,2024-06-01,Trump Hoped 'My Juror' Would Save Him From Con...
2,l6k07xn,t3_1d5a9uj,t3_1d5a9uj,True,[Full article](https://archive.ph/LMt1y),55,2024-06-01 00:16:51,June,2024-06-01,Trump Hoped 'My Juror' Would Save Him From Con...
3,l6k0abe,t3_1d5a9uj,t3_1d5a9uj,True,"> For much of the past year, lawyers to Trump ...",1109,2024-06-01 00:17:19,June,2024-06-01,Trump Hoped 'My Juror' Would Save Him From Con...
4,l6k0stl,t3_1d5a9uj,t3_1d5a9uj,True,What are the odds that something comes to ligh...,10411,2024-06-01 00:21:02,June,2024-06-01,Trump Hoped 'My Juror' Would Save Him From Con...


In [8]:
print('Comments per month:')
print(df_comments['month'].value_counts().reindex(
    ['June','July','August','September','October','November']
))
print()
print('Top-level vs nested:')
vc = df_comments['is_top_level'].value_counts()
vc.index = ['Top-level' if x else 'Nested' for x in vc.index]
print(vc)

Comments per month:
month
June          576409
July         1737467
August       1251954
September     997362
October      1107000
November      982030
Name: count, dtype: int64

Top-level vs nested:
Nested       4616527
Top-level    2035695
Name: count, dtype: int64


## 5. Verify the Join

Pick a random post and show its comments to confirm the join is correct.

In [9]:
sample_post = df_posts.sample(1, random_state=7).iloc[0]
post_comments = df_comments[df_comments['post_id'] == sample_post['name']]

print(f'Post:   {sample_post["title"]}')
print(f'Date:   {sample_post["date"].date()}')
print(f'Score:  {sample_post["score"]}')
print(f'Comments found: {len(post_comments)}')
print()
print('Top 5 comments by score:')
for _, c in post_comments.nlargest(5, 'score').iterrows():
    level = 'top-level' if c['is_top_level'] else 'nested  '
    print(f'  [{level} | score={c["score"]:4}] {c["body"][:100]}')

Post:   Hidden-camera video shows Project 2025 co-author discussing his secret work preparing for a second Trump term
Date:   2024-08-15
Score:  8
Comments found: 3

Top 5 comments by score:
  [top-level | score=   2] Don't worry guys. Trump (Famous for his honesty) said he doesn't know anything about it and has a se
  [top-level | score=   1] 
As a reminder, this subreddit [is for civil discussion.](/r/politics/wiki/index#wiki_be_civil)

In 
  [top-level | score=   1] Hi `i-am-sancho`. Thank you for participating in /r/Politics. However, your submission has been remo


## 6. Save to Parquet

Saved to `Datasources/` alongside the raw JSONL files.  
Parquet preserves dtypes (datetime, bool, int) and is ~5x smaller than CSV with ~10x faster load times.

> These files are excluded from GitHub via `.gitignore`.  
> Share via Google Drive with teammates -- they go in the `Datasources/` folder.

In [10]:
import os

posts_out    = DATA_DIR / 'df_posts_filtered.parquet'
comments_out = DATA_DIR / 'df_comments_filtered.parquet'

df_posts.to_parquet(posts_out, index=False)
df_comments.to_parquet(comments_out, index=False)

print(f'Saved posts    -> Datasources/{posts_out.name}  ({os.path.getsize(posts_out)/1e6:.1f} MB)')
print(f'Saved comments -> Datasources/{comments_out.name}  ({os.path.getsize(comments_out)/1e6:.1f} MB)')

Saved posts    -> Datasources/df_posts_filtered.parquet  (4.8 MB)
Saved comments -> Datasources/df_comments_filtered.parquet  (905.9 MB)
